In [ ]:
import json
from transformers import AutoTokenizer

from prompts import generate_single_step_fixed_system_prompt, evaluate_system_prompt_premature_attribution

# -------------------------------------------------------------------------
# [1] 토크나이저 설정 (사용하신 모델에 맞춰 변경)
# -------------------------------------------------------------------------
# 예: meta-llama/Meta-Llama-3.1-8B-Instruct
MODEL_ID = "Qwen2.5-7B-Instruct"

print(f"Loading Tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def calculate_cached_token_usage(logs):
    """
    제공된 Inference 코드의 로직을 엄밀하게 재현하여
    vLLM Prefix Caching이 적용된 실제 연산 비용(Compute Tokens)을 계산합니다.
    """
    
    # 통계 변수
    total_gen_compute = 0   # Generator 실제 연산량
    total_eval_compute = 0  # Evaluator 실제 연산량
    total_output_tokens = 0 # 생성 토큰 수
    
    # 비교용 (캐싱 없었을 때)
    raw_gen_input = 0
    raw_eval_input = 0

    for log in logs:
        # ---------------------------------------------------------------------
        # [Cache Session Init]
        # 각 문제(Sample)마다 모델별 KV Cache는 독립적으로 동작한다고 가정
        # (vLLM은 요청 간 공통 Prefix를 자동으로 감지하므로, Last Prompt ID를 추적하여 시뮬레이션)
        # ---------------------------------------------------------------------
        gen_last_prompt_ids = []
        eval_last_prompt_ids = []

        # 메타데이터 추출
        question = log['meta_data']['question']
        passages = log['meta_data']['retrieved_passages']
        
        # Generator용 Passages String (Generator Loop의 passages_str 로직)
        passages_str_gen = '\n'.join([f"Passage {i+1}: {p}" for i, p in enumerate(passages)])
        
        # Evaluator용 Passages String (Evaluator Loop의 context_str 로직)
        # 코드: context_str = '\n'.join(...).strip()
        passages_str_eval = passages_str_gen.strip()

        # 현재까지 누적된 Reasoning Steps (Generator용 상태)
        current_step_texts = []

        # Steps 순회
        for step_log in log['steps_history']:
            step_num = step_log['step_num']
            
            # Attempts 순회 (Retry 포함)
            for i, attempt in enumerate(step_log['attempts']):
                retry_index = attempt['retry_index']
                generated_text = attempt['generated_text']

                # =============================================================
                # 1. Generator Prompt Reconstruction (Rigorous)
                # =============================================================
                
                # (A) Previous Reasoning Steps String
                # 코드: previous_steps_str = '\n'.join(state["step_texts"]) if state["step_texts"] else "(No previous steps.)"
                if current_step_texts:
                    prev_steps_str_gen = '\n'.join(current_step_texts)
                else:
                    prev_steps_str_gen = "(No previous steps.)"

                # (B) Feedback String Logic
                # 코드 로직 역추적:
                # - retry_index == 0 이면 항상 "Status: N/A ..." (last_feedback이 None이거나 지워짐)
                # - retry_index > 0 이면 이전 attempt의 평가 결과 사용
                feedback_str = ""
                
                if retry_index == 0:
                    feedback_str = "Status: N/A (First attempt at this step)"
                else:
                    # 이전 시도(attempt i-1)의 결과 가져오기
                    prev_attempt = step_log['attempts'][i-1]
                    prev_eval = prev_attempt['evaluation']
                    
                    err = prev_eval.get("error_type", "Unknown")
                    diag = prev_eval.get("diagnosis", "None")
                    guid = prev_eval.get("guidance", "Proceed logically.")
                    
                    # 코드: if err == "Correct (No Error)": ... else: ...
                    # (보통 Correct면 Accepted 되어 loop를 빠져나가지만, 
                    # 만약 Correct인데도 포맷 이슈 등으로 Retry가 발생했다면 [Previous Step Was Correct]가 뜸)
                    if err == "Correct (No Error)":
                        feedback_str = f"""[Previous Step Was Correct]
- Error Type: {err}
- Diagnosis: {diag}
- Guidance: {guid}"""
                    else:
                        feedback_str = f"""[Previous Step Failed]
- Error Type: {err}
- Diagnosis: {diag}
- Guidance: {guid}"""

                # (C) Construct Full User Prompt (f-string 포맷 100% 일치)
                # 코드: prompt_user = f"""Question:\n{state['question']}\n\nRetrieved Passages: ..."""
                gen_user_content = f"""Question:
{question}

Retrieved Passages:
{passages_str_gen}

Previous Reasoning Steps:
{prev_steps_str_gen}

Feedback on Last Step:
{feedback_str}

Generate next step (start with `Step {len(current_step_texts) + 1}:`)"""

                # (D) Apply Chat Template & Tokenize
                gen_messages = [
                    {"role": "system", "content": generate_single_step_fixed_system_prompt},
                    {"role": "user", "content": gen_user_content}
                ]
                # add_generation_prompt=True 적용
                gen_prompt_ids = tokenizer.apply_chat_template(gen_messages, add_generation_prompt=True, tokenize=True)

                # (E) Calculate Generator Compute (Prefix Caching)
                common_len = 0
                min_len = min(len(gen_prompt_ids), len(gen_last_prompt_ids))
                for k in range(min_len):
                    if gen_prompt_ids[k] == gen_last_prompt_ids[k]:
                        common_len += 1
                    else:
                        break
                
                gen_input_len = len(gen_prompt_ids)
                gen_compute_len = gen_input_len - common_len
                gen_output_len = len(tokenizer.encode(generated_text))

                # Update Stats
                total_gen_compute += gen_compute_len
                total_output_tokens += gen_output_len
                raw_gen_input += gen_input_len
                
                # Update Cache History
                gen_last_prompt_ids = gen_prompt_ids

                # =============================================================
                # 2. Evaluator Prompt Reconstruction (Rigorous)
                # =============================================================
                
                # (A) Previous Steps String
                # 코드: previous_steps_str = '\n'.join(step_texts).strip()
                # (주의: Generator와 달리 else 분기 없이 바로 strip() 사용)
                prev_steps_str_eval = '\n'.join(current_step_texts).strip()

                # (B) Construct Full User Prompt
                # 코드: user_content = f"""### Task: Evaluate ...""".strip()
                eval_user_content = f"""### Task: Evaluate the Correctness of the Reasoning Step

Question:
{question}

Retrieved Passages:
{passages_str_eval}

Previous Steps:
{prev_steps_str_eval}

Step to evaluate:
{generated_text}""".strip()

                # (C) Apply Chat Template & Tokenize
                eval_messages = [
                    {"role": "system", "content": evaluate_system_prompt_premature_attribution},
                    {"role": "user", "content": eval_user_content}
                ]
                eval_prompt_ids = tokenizer.apply_chat_template(eval_messages, add_generation_prompt=True, tokenize=True)

                # (D) Calculate Evaluator Compute (Prefix Caching)
                common_len_eval = 0
                min_len_eval = min(len(eval_prompt_ids), len(eval_last_prompt_ids))
                for k in range(min_len_eval):
                    if eval_prompt_ids[k] == eval_last_prompt_ids[k]:
                        common_len_eval += 1
                    else:
                        break
                
                eval_input_len = len(eval_prompt_ids)
                eval_compute_len = eval_input_len - common_len_eval
                
                # Evaluator Output Tokens (raw_response 활용)
                if 'raw_response' in attempt['evaluation']:
                    eval_output_text = attempt['evaluation']['raw_response']
                else:
                    eval_output_text = json.dumps(attempt['evaluation']) # Fallback
                eval_output_len = len(tokenizer.encode(eval_output_text))

                # Update Stats
                total_eval_compute += eval_compute_len
                total_output_tokens += eval_output_len
                raw_eval_input += eval_input_len

                # Update Cache History
                eval_last_prompt_ids = eval_prompt_ids

                # =============================================================
                # [Loop Maintenance]
                # Accepted된 경우에만 History에 추가 (다음 Step의 Prompt에 반영됨)
                # =============================================================
                if attempt['result'] == "Accepted":
                    current_step_texts.append(generated_text)

    # 결과 집계 및 반환
    total_input_compute = total_gen_compute + total_eval_compute
    
    # 1. Proposed (With Caching) 비용: 실제 연산된 입력(Cached) + 출력
    total_final_cost_proposed = total_input_compute + total_output_tokens
    
    # 2. Baseline (No Cache) 비용: 전체 입력(Raw) + 출력
    raw_input_baseline = raw_gen_input + raw_eval_input
    total_final_cost_baseline = raw_input_baseline + total_output_tokens

    return {
        "total_cached": total_final_cost_proposed,
        "total_baseline": total_final_cost_baseline,
        "total_input_cached": total_input_compute,
        "generator_input_cached": total_gen_compute,
        "evaluator_input_cached": total_eval_compute,
        "total_input_baseline": raw_input_baseline,
        "total_output_tokens": total_output_tokens,
        "saved_cost": f"{(1 - total_final_cost_proposed / total_final_cost_baseline) * 100:.4f}%",
    }

In [ ]:
import pandas as pd

all_results = []
for model in ["gemma12b", "llama8b", "qwen7b"]:
	for dataset in ["2wiki", "hotpotqa", "musique"]:
		try:
			with open(f"/workspace/daeyong/inference_results/dev_kg_correct_1ksample_2wiki_ver3_newprompt_v2_qwen2.5_7b_2wiki_added_ver3_checkpoint_200/{model}_{dataset}_logs.json", "r") as f:
				json_data = json.load(f)

			# 이전에 작성한 엄밀한 계산 함수 호출
			stats = calculate_cached_token_usage(json_data)
			
			# 모델과 데이터셋 정보를 결과에 추가
			stats["Model"] = model
			stats["Dataset"] = dataset
			all_results.append(stats)
			
		except FileNotFoundError:
			print(f"⚠️ 파일을 찾을 수 없음")
		except Exception as e:
			print(f"⚠️ {model}_{dataset} 처리 중 에러 발생: {e}")

# 리스트를 DataFrame으로 변환
df_stats = pd.DataFrame(all_results)

# 보기 좋게 컬럼 순서 재배치 (Model, Dataset을 앞으로)
cols = ["Model", "Dataset"] + [c for c in df_stats.columns if c not in ["Model", "Dataset"]]
df_stats = df_stats[cols]
df_stats

In [ ]:
# 변환할 컬럼 리스트 정의
cost_columns = [
    'total_cached', 'total_baseline', 'total_input_cached', 
    'generator_input_cached', 'evaluator_input_cached', 
    'total_input_baseline', 'total_output_tokens'
]

# 1,000,000으로 나누고 소수점 3째 자리까지 반올림
df_stats[cost_columns] = (df_stats[cost_columns] / 1_000_000).round(3)
df_stats

In [ ]:
df_stats.to_csv("inference_cost_summary.csv", index=False)